# Tree Models
### The following models will be tested
- Random Forest
- XGBoost

In [13]:
import pandas as pd
import numpy as np
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import gc

### Load and Clean Data

In [10]:
print("--- Loading Data ---")

# A. Load Sample Data
raw_train_sample_full = pd.read_csv('train_sample_1.csv')
raw_train_sample = raw_train_sample_full.sample(frac = 0.5, random_state=42)

# B. Load Full Data & Test Data
raw_train_full = pd.read_csv('train_1.csv')
raw_test = pd.read_csv('test_1.csv')

--- Loading Data ---


In [11]:
# Clean data
cols_to_drop = [
    'trips_ended', 'net_flow', 
    'station_id', 'station_name', 
    'date', 'datetime'
]

# Drop unnecessary columns
clean_train_sample = raw_train_sample.drop(columns=cols_to_drop, errors='ignore')
clean_train_full = raw_train_full.drop(columns=cols_to_drop, errors='ignore')
clean_test = raw_test.drop(columns=cols_to_drop, errors='ignore')

print(f"Sample Train Shape: {clean_train_sample.shape}")
print(f"Full Train Shape:   {clean_train_full.shape}")
print(f"Test Shape:   {clean_test.shape}")

Sample Train Shape: (640152, 42)
Full Train Shape:   (12803054, 42)
Test Shape:   (3174974, 42)


### Preprocess Data

In [14]:
# Separate Target (y) and Features (X)
# Sample
X_train_sample = clean_train_sample.drop(columns='trips_started')
y_train_sample = clean_train_sample['trips_started']

# Full
X_train_full = clean_train_full.drop(columns='trips_started')
y_train_full = clean_train_full['trips_started']

# Test
X_test = clean_test.drop(columns='trips_started')
y_test = clean_test['trips_started']

print(f"Features used for RF: {X_train_sample.shape[1]}")

Features used for RF: 41


## Random Forest

### Hyperparameter Tuning

In [ ]:
print("\n--- Tuning Random Forest on Sample Data ---")

# Define the grid
param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, 25],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4],
    'max_features': ['sqrt', 0.5]
}

rf_tuner = RandomForestRegressor(random_state=42, n_jobs=6) # Changed to use 6 cores to limit RAM usage

# Run Random Search
random_search = RandomizedSearchCV(
    estimator=rf_tuner,
    param_distributions=param_dist,
    n_iter=10, 
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    random_state=42,
    n_jobs=6 # Changed to use 6 cores to limit RAM usage
)

random_search.fit(X_train_sample, y_train_sample)
best_params_rf = random_search.best_params_
print(f"Best Parameters: {best_params_rf}")

### Training and Evaluation

In [ ]:
# --- Model A: Trained on Sample Data ---
print("\n--- Training RF on Sample Data ---")
rf_model_sample = RandomForestRegressor(
    **best_params_rf, 
    random_state=42, 
    n_jobs=-1
)
rf_model_sample.fit(X_train_sample, y_train_sample)

# Predict on Test
y_pred_sample_on_test = rf_model_sample.predict(X_test)

# --- Model B: Trained on Full Data ---
print("\n--- Training RF on Full Data ---")
rf_model_full = RandomForestRegressor(
    **best_params_rf, 
    random_state=42, 
    n_jobs=-1
)
rf_model_full.fit(X_train_full, y_train_full)

# Predict on Test
y_pred_full_on_test = rf_model_full.predict(X_test)

In [ ]:
rmse_sample_rf = np.sqrt(mean_squared_error(y_test, y_pred_sample_on_test))
r2_sample_rf = r2_score(y_test, y_pred_sample_on_test)

rmse_full_rf = np.sqrt(mean_squared_error(y_test, y_pred_full_on_test))
r2_full_rf = r2_score(y_test, y_pred_full_on_test)

print("\n" + "="*60)
print(f"{'METRIC':<10} | {'RF (Sample Trained)':<20} | {'RF (Full Trained)':<20}")
print("="*60)
print(f"{'Test RMSE':<10} | {rmse_sample_rf:<20.4f} | {rmse_full_rf:<20.4f}")
print(f"{'Test R^2':<10} | {r2_sample_rf:<20.4f} | {r2_full_rf:<20.4f}")
print("="*60)

## XGBoost

### Device Setup

In [ ]:
# 1. Detect Device
# We use PyTorch to check for Nvidia GPU (CUDA) availability
if torch.cuda.is_available():
    print(f"NVIDIA GPU Detected: {torch.cuda.get_device_name(0)}")
    target_device = "cuda"
    n_jobs_xgb = 1  # GPU handles the parallelism internally, so we set CPU jobs to 1
else:
    print("No NVIDIA GPU found. Using CPU.")
    target_device = "cpu"
    n_jobs_xgb = 6  # Set to 4 or 6 based on your 25GB RAM

print(f"Models will run on: {target_device.upper()}")

### Hyperparameter Tuning

In [ ]:
print("\n--- Tuning XGBoost on Sample Data ---")

# XGBoost specific hyperparameters
param_dist = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],   # Lower rate + more trees usually wins
    'max_depth': [6, 10, 15],             # XGBoost prefers shallower trees than RF
    'subsample': [0.6, 0.8, 1.0],         # Row sampling to prevent overfitting
    'colsample_bytree': [0.6, 0.8, 1.0],   # Feature sampling (like max_features in RF)
    'gamma': [0, 0.5, 1],                   # minimum loss reduction to make a split
    'reg_lambda': [1, 5, 10]                # L2 regularization strength
}

xgb_tuner = XGBRegressor(
    target_device=target_device,
    random_state=42, 
    n_jobs=n_jobs_xgb,
    objective='reg:squarederror' # Explicitly set objective for regression
)

random_search = RandomizedSearchCV(
    estimator=xgb_tuner,
    param_distributions=param_dist,
    n_iter=15,           # XGB is faster than RF, so we can try a few more combos
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_sample, y_train_sample)
best_params_xgb = random_search.best_params_
print(f"Best Parameters: {best_params_xgb}")

### Training and Evaluation

In [ ]:
# --- Model A: Trained on Sample Data ---
print("\n--- Training XGBoost on Sample Data ---")
xgb_model_sample = XGBRegressor(
    **best_params_xgb,
    random_state=42,
    n_jobs=-1, 
    objective='reg:squarederror'
)
xgb_model_sample.fit(X_train_sample, y_train_sample)

# Predict on Test
y_pred_sample_on_test = xgb_model_sample.predict(X_test)

# --- Model B: Trained on Full Data ---
print("\n--- Training XGBoost on Full Data ---")
xgb_model_full = XGBRegressor(
    **best_params_xgb, 
    random_state=42, 
    n_jobs=-1, 
    objective='reg:squarederror'
)
xgb_model_full.fit(X_train_full, y_train_full)

# Predict on Test
y_pred_full_on_test = xgb_model_full.predict(X_test)

In [ ]:
rmse_sample_xgb = np.sqrt(mean_squared_error(y_test, y_pred_sample_on_test))
r2_sample_xgb = r2_score(y_test, y_pred_sample_on_test)

rmse_full_xgb = np.sqrt(mean_squared_error(y_test, y_pred_full_on_test))
r2_full_xgb = r2_score(y_test, y_pred_full_on_test)

print("\n" + "="*60)
print(f"{'METRIC':<10} | {'XGB (Sample Trained)':<20} | {'XGB (Full Trained)':<20}")
print("="*60)
print(f"{'Test RMSE':<10} | {rmse_sample_xgb:<20.4f} | {rmse_full_xgb:<20.4f}")
print(f"{'Test R^2':<10} | {r2_sample_xgb:<20.4f} | {r2_full_xgb:<20.4f}")
print("="*60)

In [ ]:
# Clean up RAM
del raw_train_full, clean_train_full, X_train_full, y_train_full
gc.collect()